<a href="https://colab.research.google.com/github/Aradhyay/satellite_vision_transfer/blob/main/Satellite_Imagery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
!pip install patchify

# !pip install rasterio

In [ ]:
import os # it is used to interact with virtual machine's file system.
import cv2 # OpenCV for image processing
from PIL import Image # image class from PILLOW library is used to read and manipulate the images using python code
import numpy as np
from patchify import patchify # it is used to split large images into patches
from sklearn.preprocessing import MinMaxScaler, StandardScaler # MinMaxScaler is used to scale an image to a range and StandardScaler removes mean and scales to unit variance
from matplotlib import pyplot as plt # it is used to plot the predictions
import random

In [ ]:
minmaxscaler=MinMaxScaler()

In [ ]:
!ls -lah '/content/drive/MyDrive/Colab Notebooks/Satellite-Data/Dubai-Dataset'

In [ ]:
dataset_root_folder="/content/drive/MyDrive/Colab Notebooks/Satellite-Data/"

In [ ]:
dataset_name="Dubai-Dataset"

In [ ]:
for path , subdirs , files in os.walk(dataset_root_folder):
  dir_name = path.split(os.path.sep)[-1]
  print(dir_name)
  if dir_name == 'images':
    images=os.listdir(path)
    print(images)

In [ ]:
image=cv2.imread(f'{dataset_root_folder}/{dataset_name}/Tile 1/images/image_part_001.jpg',1) # reading the image from the path

In [ ]:
type(Image.fromarray(image))

In [ ]:
image.shape

In [ ]:
image_patch_size=256

In [ ]:
(image.shape[1]//image_patch_size)*image_patch_size

In [ ]:
image_patches=patchify(image,(image_patch_size,image_patch_size,3), step=image_patch_size)

In [ ]:
print(image_patches.shape)

In [ ]:
image_x=image_patches[0,0,:,:]

In [ ]:
image_y=minmaxscaler.fit_transform(image_x.reshape(-1,image_x.shape[-1])).reshape(image_x.shape)

print(image_y.shape)

In [ ]:
len(image_patches)

In [ ]:
image_dataset=[]
mask_dataset=[]
for image_type in ['images','masks']:
  if image_type=='images':
    image_extension='jpg'
  elif image_type=='masks':
    image_extension='png'
  for tile_id in range(1,8):
    for image_id in range(1,10):
      image=cv2.imread(f'{dataset_root_folder}{dataset_name}/Tile {tile_id}/{image_type}/image_part_00{image_id}.{image_extension}',1)
      if image is not None:
        if image_type=='masks':
          image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        size_x=(image.shape[1]//image_patch_size)*image_patch_size
        size_y=(image.shape[0]//image_patch_size)*image_patch_size
        image=Image.fromarray(image)
        image=image.crop((0,0,size_x,size_y))
        image=np.array(image)
        patched_images=patchify(image,(image_patch_size,image_patch_size,3), step=image_patch_size)
        for i in range(patched_images.shape[0]):
          for j in range (patched_images.shape[1]):
            individual_patched_image=patched_images[i,j,0,:,:,:]
            if image_type=='images':
              individual_patched_image=minmaxscaler.fit_transform(individual_patched_image.reshape(-1,individual_patched_image.shape[-1])).reshape(individual_patched_image.shape)
              image_dataset.append(individual_patched_image)
            elif image_type=='masks':
              individual_patched_mask=patched_images[i,j,0,:,:,:]
              mask_dataset.append(individual_patched_mask)

In [ ]:
print(len(image_dataset))
print(len(mask_dataset))

In [ ]:
image_dataset=np.array(image_dataset)
mask_dataset=np.array(mask_dataset)

In [ ]:
type(image_dataset[0])

In [ ]:
plt.figure(figsize=(10,8))
plt.subplot(121)
plt.imshow(image_dataset[0])
plt.subplot(122)
plt.imshow(mask_dataset[0])

In [ ]:
class_building ='#3c1098'
class_building=class_building.lstrip('#')
class_building=np.array(tuple(int(class_building[i:i+2],16)for i in (0,2,4)))
print(class_building)

In [ ]:
class_building ='#3c1098'
class_building=class_building.lstrip('#')
class_building=np.array(tuple(int(class_building[i:i+2],16)for i in (0,2,4)))
print(class_building)

class_land ='#8429F6'
class_land=class_land.lstrip('#')
class_land=np.array(tuple(int(class_land[i:i+2],16)for i in (0,2,4)))
print(class_land)

class_road ='#6EC1E4'
class_road=class_road.lstrip('#')
class_road=np.array(tuple(int(class_road[i:i+2],16)for i in (0,2,4)))
print(class_road)

class_vegetation ='#FEDD3A'
class_vegetation=class_vegetation.lstrip('#')
class_vegetation=np.array(tuple(int(class_vegetation[i:i+2],16)for i in (0,2,4)))
print(class_vegetation)

class_water ='#E2A929'
class_water=class_water.lstrip('#')
class_water=np.array(tuple(int(class_water[i:i+2],16)for i in (0,2,4)))
print(class_water)

class_unlabelled ='#9B9B9B'
class_unlabelled=class_unlabelled.lstrip('#')
class_unlabelled=np.array(tuple(int(class_unlabelled[i:i+2],16)for i in (0,2,4)))
print(class_unlabelled)

In [ ]:
def rgb_to_label(label):
  label_segment=np.zeros(label.shape, dtype=np.uint8)
  label_segment[np.all(label==class_water, axis=-1)]=0
  label_segment[np.all(label==class_land, axis=-1)]=1
  label_segment[np.all(label==class_road, axis=-1)]=2
  label_segment[np.all(label==class_building, axis=-1)]=3
  label_segment[np.all(label==class_vegetation, axis=-1)]=4
  label_segment[np.all(label==class_unlabelled, axis=-1)]=5
  label_segment=label_segment[:,:,0]
  print(label_segment)
  return label_segment

In [ ]:
labels=[]
for i in range(mask_dataset.shape[0]):
  label=rgb_to_label(mask_dataset[i])
  labels.append(label)

In [ ]:
print(len(labels))

In [ ]:
labels=np.array(labels)

In [ ]:
labels[3]

In [ ]:
master_training_dataset=image_dataset

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(master_training_dataset, labels, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
image_height=X_train.shape[1]
image_width=X_train.shape[2]
image_channel=X_train.shape[3]
print(image_height, image_width, image_channel)

In [ ]:
!pip install segmentation-models-pytorch torchmetrics

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class SatelliteDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = self.masks[idx]

        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        if img_tensor.max() > 1.0:
            img_tensor = img_tensor / 255.0

        return img_tensor, mask_tensor

train_dataset = SatelliteDataset(X_train, y_train)
test_dataset = SatelliteDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"epoch: {len(train_loader)}")

In [ ]:
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,
)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

dice_fn = DiceLoss(mode='multiclass')
focal_fn = FocalLoss(mode='multiclass')

In [ ]:
epochs = 65

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device).long().squeeze(1)

        optimizer.zero_grad()
        predictions = model(images)
        loss = dice_fn(predictions, masks) + focal_fn(predictions, masks)
        loss.backward()
        optimizer.step()
        train_loss =train_loss + loss.item()

    avg_loss = train_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Combined Loss: {avg_loss:.4f}")

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex

iou_metric_per_class = MulticlassJaccardIndex(num_classes=6, average='none').to(device)

model.eval()

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device).long().squeeze(1)

        raw_predictions = model(images)
        prob_predictions = torch.softmax(raw_predictions, dim=1)
        pred_classes = torch.argmax(prob_predictions, dim=1)

        iou_metric_per_class.update(pred_classes, masks)

per_class_iou = iou_metric_per_class.compute()

class_names = ["Unlabeled", "Building", "Land", "Road", "Vegetation", "Water"]

print("Per-Class IoU Scores ")
for name, score in zip(class_names, per_class_iou):
    print(f"{name}: {score:.4f}")

print(f"\n Overall Average (mIoU): {per_class_iou.mean():.4f}")

iou_metric_per_class.reset()

In [ ]:
# Saving the model.
# torch.save(model.state_dict(), 'dubai_6class_model_final_unet.pth')
# print("Model saved successfully.")

In [ ]:
import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=6,
)

model.load_state_dict(torch.load('dubai_6class_model_unet.pth', map_location=device))
model = model.to(device)

model.eval()
print("6-Class Dubai Model successfully loaded")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

color_map = np.array([
    [155, 155, 155], # 0: Unlabeled (#9B9B9B)
    [60,  16,  152], # 1: Building (#3C1098)
    [132, 41,  246], # 2: Land (#8429F6)
    [110, 193, 228], # 3: Road (#6EC1E4)
    [254, 221, 58],  # 4: Vegetation (#FEDD3A)
    [226, 169, 41]   # 5: Water (#E2A929)
], dtype=np.uint8)

model.eval()
images, true_masks = next(iter(test_loader))

single_image = images[7].unsqueeze(0).to(device)
single_mask = true_masks[7].squeeze().numpy()

with torch.no_grad():
    raw_pred = model(single_image)
    prob_pred = torch.softmax(raw_pred, dim=1)
    pred_class = torch.argmax(prob_pred, dim=1).cpu().squeeze().numpy()

rgb_true_mask = color_map[single_mask.astype(int)]
rgb_pred_mask = color_map[pred_class.astype(int)]

fig,arr = plt.subplots(1, 3,figsize=(10,8))

img_display = single_image[0].cpu().permute(1, 2, 0).numpy()

arr[0].imshow(img_display)
arr[0].set_title("Original Dubai Image")
arr[0].axis('off')

arr[1].imshow(rgb_true_mask)
arr[1].set_title("True Ground Mask")
arr[1].axis('off')

arr[2].imshow(rgb_pred_mask)
arr[2].set_title("Model's Prediction")
arr[2].axis('off')

plt.show()

In [ ]:
import zipfile
import os

zip_path = 'mumbai.zip'
extract_path = './mumbai_data'

print("Unzipping")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("unzipping complete")

In [ ]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import os

class MumbaiDirectoryDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.items = []
        mask_files = os.listdir(masks_dir)
        mask_dict = {os.path.splitext(f)[0]: os.path.join(masks_dir, f) for f in mask_files}

        for img_name in sorted(os.listdir(images_dir)):
            base_name = os.path.splitext(img_name)[0]
            if base_name in mask_dict:
                img_path = os.path.join(images_dir, img_name)
                mask_path = mask_dict[base_name]
                self.items.append((img_path, mask_path))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, mask_path = self.items[idx]

        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask_rgb = cv2.cvtColor(cv2.imread(mask_path), cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (256, 256))
        mask_rgb = cv2.resize(mask_rgb, (256, 256), interpolation=cv2.INTER_NEAREST)

        integer_mask = np.zeros((256, 256), dtype=np.int64)

        integer_mask[(mask_rgb == [200, 200, 200]).all(axis=-1)] = 1
        integer_mask[(mask_rgb == [250, 235, 185]).all(axis=-1)] = 1
        integer_mask[(mask_rgb == [200, 160, 40]).all(axis=-1)] = 2
        integer_mask[(mask_rgb == [100, 100, 150]).all(axis=-1)] = 3
        integer_mask[(mask_rgb == [80, 140, 50]).all(axis=-1)] = 4
        integer_mask[(mask_rgb == [40, 120, 240]).all(axis=-1)] = 5

        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        mask_tensor = torch.tensor(integer_mask, dtype=torch.long)

        return img_tensor, mask_tensor

mumbai_test_images = './mumbai_data/Dataset/Prepared_Dataset/test/images'
mumbai_test_masks = './mumbai_data/Dataset/Prepared_Dataset/test/masks'
mumbai_test_dataset = MumbaiDirectoryDataset(mumbai_test_images, mumbai_test_masks)
mumbai_test_loader = DataLoader(mumbai_test_dataset, batch_size=16, shuffle=False)

print("ready for testing")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

color_map = np.array([
    [155, 155, 155], # 0: Unlabeled
    [60,  16,  152], # 1: Building
    [132, 41,  246], # 2: Land
    [110, 193, 228], # 3: Road
    [254, 221, 58],  # 4: Vegetation
    [226, 169, 41]   # 5: Water
], dtype=np.uint8)

model.eval()
images, true_masks = next(iter(mumbai_test_loader))

idx = 15
single_image = images[idx].unsqueeze(0).to(device)
single_mask = true_masks[idx].numpy()

print(f"Unique pixel values in this Mumbai mask: {np.unique(single_mask)}")

with torch.no_grad():
    raw_pred = model(single_image)
    prob_pred = torch.softmax(raw_pred, dim=1)
    pred_class = torch.argmax(prob_pred, dim=1).cpu().squeeze().numpy()

rgb_pred_mask = color_map[pred_class.astype(int)]

fig,arr = plt.subplots(1, 3,figsize=(10,8))

img_display = single_image[0].cpu().permute(1, 2, 0).numpy()

arr[0].imshow(img_display)
arr[0].set_title("Unseen Mumbai Image")
arr[0].axis('off')

arr[1].imshow(single_mask, cmap='viridis')
arr[1].set_title("True Mumbai Ground Mask")
arr[1].axis('off')

arr[2].imshow(rgb_pred_mask)
arr[2].set_title("Dubai Model's Prediction")
arr[2].axis('off')

plt.show()

In [ ]:
from google.colab import files
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

print("Upload the Satellite Image")
uploaded_img = files.upload()
img_filename = list(uploaded_img.keys())[0]

print("\nUpload the Corresponding Mask")
uploaded_mask = files.upload()
mask_filename = list(uploaded_mask.keys())[0]

img = cv2.imread(img_filename)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img, (256, 256))

img_tensor = torch.tensor(img_resized, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0
img_tensor = img_tensor.to(device)

mask = cv2.imread(mask_filename)
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
mask_resized = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

model.eval()
with torch.no_grad():
    raw_pred = model(img_tensor)
    prob_pred = torch.softmax(raw_pred, dim=1)
    pred_class = torch.argmax(prob_pred, dim=1).cpu().squeeze().numpy()

color_map = np.array([
    [155, 155, 155], # 0: Unlabeled
    [60,  16,  152], # 1: Building
    [132, 41,  246], # 2: Land
    [110, 193, 228], # 3: Road
    [254, 221, 58],  # 4: Vegetation
    [226, 169, 41]   # 5: Water
], dtype=np.uint8)

rgb_pred_mask = color_map[pred_class.astype(int)]

fig, arr = plt.subplots(1, 3, figsize=(18, 6))

arr[0].imshow(img_resized)
arr[0].set_title("Uploaded Satellite Image", fontsize=14)
arr[0].axis('off')

arr[1].imshow(mask_resized)
arr[1].set_title("Ground Truth Mask", fontsize=14)
arr[1].axis('off')

arr[2].imshow(rgb_pred_mask)
arr[2].set_title("Model Prediction", fontsize=14)
arr[2].axis('off')

plt.tight_layout()
plt.show()